# 牛牛战法 Round36–37：主线确认后的持仓升级复核

本笔记本复核两类账户级动作：Round36 在题材跃迁后把试仓切换为启动退出语义；Round37 只加仓至总仓位上限 10%，保留试仓退出语义。所有信号均来自当日及以前可见的完整题材横截面，交易在下一交易日开盘执行。

In [1]:
import json
import os
from pathlib import Path

from IPython.display import display

PATHS = {
    ('round36', 'current'): Path(os.environ.get('NIUONE_ROUND36_CURRENT_RESULT', '/private/tmp/niuone-round36-holding-upgrade-current.json')),
    ('round36', 'extended'): Path(os.environ.get('NIUONE_ROUND36_EXTENDED_RESULT', '/private/tmp/niuone-round36-holding-upgrade-extended.json')),
    ('round37', 'current'): Path(os.environ.get('NIUONE_ROUND37_CURRENT_RESULT', '/private/tmp/niuone-round37-holding-scale-current.json')),
    ('round37', 'extended'): Path(os.environ.get('NIUONE_ROUND37_EXTENDED_RESULT', '/private/tmp/niuone-round37-holding-scale-extended.json')),
}
DATA = {key: json.loads(path.read_text(encoding='utf-8')) for key, path in PATHS.items()}
print({f'{round_id}_{pool}': str(path) for (round_id, pool), path in PATHS.items()})

{'round36_current': '/private/tmp/niuone-round36-holding-upgrade-current.json', 'round36_extended': '/private/tmp/niuone-round36-holding-upgrade-extended.json', 'round37_current': '/private/tmp/niuone-round37-holding-scale-current.json', 'round37_extended': '/private/tmp/niuone-round37-holding-scale-extended.json'}


In [2]:
BASELINE = 'production_lifecycle_early_recovery_lt2'
PRIMARY = ('old_sealed', 'train_a', 'train_b', 'validation')
LABELS = {
    BASELINE: 'Round29 早段影子',
    'production_lifecycle_early_recovery_lt2_upgrade_top5_persistent': '前5后切换启动',
    'production_lifecycle_early_recovery_lt2_upgrade_new_top5_persistent': '新进前5后切换启动',
    'production_lifecycle_early_recovery_lt2_scale_top5_persistent': '前5后加仓、保留退出',
    'production_lifecycle_early_recovery_lt2_scale_new_top5_persistent': '新进前5后加仓、保留退出',
}
rows = []
for (round_id, pool), data in DATA.items():
    baseline = data['candidates'][BASELINE]['development_aggregate']
    for candidate, payload in data['candidates'].items():
        aggregate = payload['development_aggregate']
        recent = payload['windows']['recent']
        add_orders = sum(payload['windows'][window]['statistics']['add_order_count'] for window in PRIMARY)
        rows.append({
            'round': round_id, 'pool': pool, 'candidate': LABELS[candidate],
            'trades': aggregate['completed_trade_count'], 'win_rate_pct': aggregate['win_rate_pct'],
            'win_rate_delta_pp': round(aggregate['win_rate_pct'] - baseline['win_rate_pct'], 4),
            'compounded_return_pct': aggregate['compounded_portfolio_return_pct'],
            'return_delta_pp': round(aggregate['compounded_portfolio_return_pct'] - baseline['compounded_portfolio_return_pct'], 4),
            'positive_windows': f"{aggregate['positive_window_count']}/{aggregate['evaluated_window_count']}",
            'worst_drawdown_pct': aggregate['worst_max_drawdown_pct'], 'add_orders': add_orders,
            'recent_win_rate_pct': recent['statistics']['win_rate_pct'],
            'recent_return_pct': recent['portfolio']['total_return_pct'],
        })
summary = rows
display(summary)

[{'round': 'round36',
  'pool': 'current',
  'candidate': 'Round29 早段影子',
  'trades': 143,
  'win_rate_pct': 59.4406,
  'win_rate_delta_pp': 0.0,
  'compounded_return_pct': 14.2912,
  'return_delta_pp': 0.0,
  'positive_windows': '4/4',
  'worst_drawdown_pct': -1.7972,
  'add_orders': 0,
  'recent_win_rate_pct': 68.75,
  'recent_return_pct': 2.2353},
 {'round': 'round36',
  'pool': 'current',
  'candidate': '前5后切换启动',
  'trades': 149,
  'win_rate_pct': 59.0604,
  'win_rate_delta_pp': -0.3802,
  'compounded_return_pct': 11.4611,
  'return_delta_pp': -2.8301,
  'positive_windows': '4/4',
  'worst_drawdown_pct': -1.7716,
  'add_orders': 22,
  'recent_win_rate_pct': 62.5,
  'recent_return_pct': 2.5637},
 {'round': 'round36',
  'pool': 'current',
  'candidate': '新进前5后切换启动',
  'trades': 148,
  'win_rate_pct': 58.7838,
  'win_rate_delta_pp': -0.6568,
  'compounded_return_pct': 11.0919,
  'return_delta_pp': -3.1993,
  'positive_windows': '4/4',
  'worst_drawdown_pct': -1.7716,
  'add_orders': 

In [3]:
def primary_rows(data, candidate):
    return [(window, row) for window in PRIMARY for row in data['candidates'][candidate]['windows'][window]['completed_trade_features']]

def trade_key(window, row):
    return window, row['symbol'], row['entry_date'], row['strategy_id']

attribution = []
for pool in ('current', 'extended'):
    data = DATA[('round36', pool)]
    baseline_by_key = {trade_key(window, row): row for window, row in primary_rows(data, BASELINE)}
    for candidate in (
        'production_lifecycle_early_recovery_lt2_upgrade_top5_persistent',
        'production_lifecycle_early_recovery_lt2_upgrade_new_top5_persistent',
    ):
        matched = []
        for window, row in primary_rows(data, candidate):
            if 'niu_emerging' not in tuple(row.get('strategy_path') or ()): continue
            baseline_row = baseline_by_key.get(trade_key(window, row))
            if baseline_row is not None: matched.append((baseline_row, row))
        attribution.append({
            'pool': pool, 'candidate': LABELS[candidate], 'matched_upgraded_positions': len(matched),
            'baseline_win_rate_pct': round(100 * sum(float(before['net_return_pct']) > 0 for before, _ in matched) / len(matched), 4),
            'upgraded_win_rate_pct': round(100 * sum(float(after['net_return_pct']) > 0 for _, after in matched) / len(matched), 4),
            'baseline_average_return_pct': round(sum(float(before['net_return_pct']) for before, _ in matched) / len(matched), 4),
            'upgraded_average_return_pct': round(sum(float(after['net_return_pct']) for _, after in matched) / len(matched), 4),
            'average_return_delta_pp': round(sum(float(after['net_return_pct']) - float(before['net_return_pct']) for before, after in matched) / len(matched), 4),
        })
display(attribution)

[{'pool': 'current',
  'candidate': '前5后切换启动',
  'matched_upgraded_positions': 20,
  'baseline_win_rate_pct': 95.0,
  'upgraded_win_rate_pct': 90.0,
  'baseline_average_return_pct': 8.9108,
  'upgraded_average_return_pct': 3.1191,
  'average_return_delta_pp': -5.7917},
 {'pool': 'current',
  'candidate': '新进前5后切换启动',
  'matched_upgraded_positions': 15,
  'baseline_win_rate_pct': 93.3333,
  'upgraded_win_rate_pct': 93.3333,
  'baseline_average_return_pct': 6.9723,
  'upgraded_average_return_pct': 2.06,
  'average_return_delta_pp': -4.9124},
 {'pool': 'extended',
  'candidate': '前5后切换启动',
  'matched_upgraded_positions': 20,
  'baseline_win_rate_pct': 95.0,
  'upgraded_win_rate_pct': 90.0,
  'baseline_average_return_pct': 10.919,
  'upgraded_average_return_pct': 3.0109,
  'average_return_delta_pp': -7.9082},
 {'pool': 'extended',
  'candidate': '新进前5后切换启动',
  'matched_upgraded_positions': 13,
  'baseline_win_rate_pct': 100.0,
  'upgraded_win_rate_pct': 100.0,
  'baseline_average_return_pc

In [4]:
for pool in ('current', 'extended'):
    baseline36 = DATA[('round36', pool)]['candidates'][BASELINE]['development_aggregate']
    baseline37 = DATA[('round37', pool)]['candidates'][BASELINE]['development_aggregate']
    assert baseline36 == baseline37
for data in DATA.values():
    for payload in data['candidates'].values():
        for window in (*PRIMARY, 'recent'):
            result = payload['windows'][window]
            assert result['statistics']['completed_trade_count'] == len(result['completed_trade_features'])
            assert result['statistics']['portfolio_return_pct'] == result['portfolio']['total_return_pct']
            assert 0 <= result['statistics']['win_rate_pct'] <= 100
            assert result['portfolio']['max_drawdown_pct'] <= 0
candidate_groups = {}
for row in summary:
    if row['candidate'] != 'Round29 早段影子': candidate_groups.setdefault(row['candidate'], []).append(row)
for candidate, group in candidate_groups.items():
    assert not all(row['win_rate_delta_pp'] > 0 and row['return_delta_pp'] > 0 for row in group), candidate
print('QUALITY CHECKS PASSED: baseline identity, row counts, portfolio metrics, win-rate bounds, and drawdown signs are consistent.')

QUALITY CHECKS PASSED: baseline identity, row counts, portfolio metrics, win-rate bounds, and drawdown signs are consistent.


## 结论

- Round36 的点时题材确认事件能识别高胜率持仓，但切换为启动退出语义显著截断赢家，因此两个候选拒绝。
- Round37 保留原退出并加仓后，宽松前 5 版本提高了复合收益，却以更低胜率、更大回撤和较弱验证窗为代价；新进前 5 版本没有提高收益，因此两个候选也拒绝。
- 生产策略保持不变；`production_lifecycle_early_recovery_lt2` 继续仅作 Round29 影子候选，等待 2026-08-03 起的严格前向证据。